In [1]:
import toml
import message_ix
import ixmp
import pandas as pd
from ixmp import Scenario
from pathlib import Path
from utilities.preprocess.setup import setup_scenario

from message_ix_models.util import (
    broadcast,
    make_matched_dfs,
    package_data_path,
    same_node,
    same_time,
)

In [2]:
scenario_name = 'hydrogen_test'

In [3]:
platform, msg = setup_scenario(scenario_name)

In [4]:
msg.set('map_shares_commodity_share')

,shares,node_share,node,type_tec,mode,commodity,level
0,share_low_carbon,Canada,Canada,powerplant_lowcarbon,M1,electr,secondary
1,share_electric_transport,Canada,Canada,electric transport,M1,transport,useful


In [ ]:
msg.check_out()

In [ ]:
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)

In [ ]:
#msg.remove_solution()

In [ ]:
#msg.check_out()

In [5]:
import toml

config_path = Path(f"scenarios/{scenario_name}/config.toml")
if not config_path.exists():
    raise FileNotFoundError(f"Config file does not exist at {config_path}")
    
config = toml.load(config_path)

In [7]:
scenario_folder = config['data']['scenario_folder']

In [9]:
msg.check_out()

In [15]:
path=Path(scenario_folder)
data = pd.ExcelFile(path / 'hydrogen_shares_V2.xlsx')

# Loop through each sheet
for sheet_name in data.sheet_names:
    # Perform operations on the DataFrame
    df = pd.read_excel(path /'hydrogen_shares_V2.xlsx', sheet_name=sheet_name)
    print(f"Sheet name: {sheet_name}")
    if sheet_name in msg.set_list():
        if sheet_name == 'cat_tec':
            continue
        print('sets',sheet_name)
        msg.add_set(sheet_name, df[sheet_name])
    elif sheet_name in msg.par_list():
        msg.add_par(sheet_name, df)
        print('par',sheet_name)

msg.commit('Updated historical activity based on new data')

print(msg.par("share_commodity_lo")['shares'].unique())

Sheet name: type_tec
sets type_tec
Sheet name: cat_tec
Sheet name: shares
sets shares
Sheet name: map_shares_commodity_share
sets map_shares_commodity_share


KeyError: 'map_shares_commodity_share'

In [16]:
msg.add_set('map_shares_commodity_share', df)

ValueError: The index set 'type_tec' does not have an element 'h2_trp'!

In [11]:
df

,shares,node_share,node,type_tec,mode,commodity,level
0,share_h2_i,Alberta,Alberta,h2_industrial,M1,i_spec,useful
1,share_h2_i,Alberta,Alberta,h2_industrial,M1,i_therm,useful
2,share_h2_i,BritishColumbia,BritishColumbia,h2_industrial,M1,i_spec,useful
3,share_h2_i,BritishColumbia,BritishColumbia,h2_industrial,M1,i_therm,useful
4,share_h2_i,Manitoba,Manitoba,h2_industrial,M1,i_spec,useful
...,...,...,...,...,...,...,...
60,share_h2_trp,Ontario,Ontario,h2_trp,M1,transport,useful
61,share_h2_trp,PrinceEdwardIsland,PrinceEdwardIsland,h2_trp,M1,transport,useful
62,share_h2_trp,Quebec,Quebec,h2_trp,M1,transport,useful
63,share_h2_trp,Saskatchewan,Saskatchewan,h2_trp,M1,transport,useful


In [ ]:

msg.solve(solve_options={'barcrossalg': '2','lpmethod':'1'}, case=caseName)

In [ ]:
caseName

In [ ]:
msg.set("map_share_commodity_lo")

In [ ]:
msg.par("share_commodity_lo")['shares'].unique()

In [ ]:
from message_ix.report import Reporter
rep = Reporter.from_scenario(msg)
report = rep.get('message::default') # in fact, only in = input x ACT, out = output x ACT is needed

In [ ]:
import numpy as np
min_year = 2020
max_year = 2060

report_df = report.timeseries()
report_df.reset_index(inplace=True)
report_df.columns = report_df.columns.astype(str)
report_df.columns = report_df.columns.str.title()

# Plotted years
years = msg.set('year').tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

# Create data for Sankey diagrams and plot it using plotly
report_df.fillna(0, inplace = True)

# Convert from GWa to TWh (*8.76)
c = report_df.select_dtypes(include=[np.number]) * 8.76
report_df[c.columns] = c

# Drop not necessary years
report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(report_df.columns.to_series()['2020': str(min_year)],
                        axis=1)

# Identify input and output variables and its flows (provided within the Reporter: in = input x ACT, out = output x ACT)
in_flows = report_df[report_df['Variable'].str.startswith("in|")]
out_flows = report_df[report_df['Variable'].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Variable"] = report_df["Variable"].str.replace('emis', 'Emissions', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|final|', 'Final Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|final|', 'Final Energy|', regex = False)
#report_df["Variable"] = report_df["Variable"].str.replace('in|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|primary|', 'Primary Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|renewable|', 'Primary Energy|renewable|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('total om cost|', 'Total Costs|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('out|useful|', 'Useful Energy|', regex = False)
report_df["Variable"] = report_df["Variable"].str.replace('in|useful|', 'Useful Energy|', regex = False)

In [ ]:
report_df.to_csv('MESSAGE_CANADA_H2_Shares_Test.csv', index = False)